In [1]:
# ── 셀 1 │ 01_Fine_tune.ipynb 의 정의를 그대로 가져온다 ──
import json, os, itertools, datetime
import numpy as np, pandas as pd, torch

NB = 'C:/Users/iisl/Desktop/0_Molecualar_Informatics/2026_final/2026_new/02_finetune/01_Fine_tune.ipynb'
_nb = json.load(open(NB, encoding='utf-8'))
_code = [''.join(c['source']) for c in _nb['cells'] if c['cell_type'] == 'code']
for i in range(1, 6):          # 셀 0(캐시 삭제)은 제외, 셀 1~5만
    exec(_code[i], globals())

print('재사용 완료 | 임상', len(cg), '| 스캐폴드', C['scaffold'].nunique(), '| device', DEV)

C:\Users\iisl\anaconda3\envs\esu_gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device cuda
임상 풀 1225  양성 725
캐시 로드 1225개
스캐폴드 836종
Broad  양성  725  음성  500  합계 1225  양성률 0.592
A      양성  133  음성  500  합계  633  양성률 0.210
B      양성  592  음성  500  합계 1092  양성률 0.542
C      양성  133  음성  592  합계  725  양성률 0.183
모델·loss 준비 완료. 파라미터 88,066
재사용 완료 | 임상 1225 | 스캐폴드 836 | device cuda


In [2]:
# ── 셀 2 │ 설정을 인자로 받는 학습 (검증 AUROC만 반환, 테스트 미사용) ──
from torch_geometric.loader import DataLoader

BASE = dict(lr_bb=1e-4, lr_clf=5e-3, drop=0.3, wd=5e-4, bs=32)

def fit_cfg(gs_tr, lab_tr, gs_va, lab_va, ckpt, seed, cfg):
    seed_all(seed)
    model = GAT_SI(ATOM_DIM, hc=HID, h=HEADS, d=cfg['drop']).to(DEV)
    if ckpt is not None:
        model = load_backbone(model, ckpt).to(DEV)
    crit = make_loss(LOSS, np.asarray(lab_tr))
    opt = torch.optim.Adam(
        [{'params': model.backbone_parameters(), 'lr': cfg['lr_bb']},
         {'params': model.clf.parameters(),      'lr': cfg['lr_clf']}],
        weight_decay=cfg['wd'])
    L_tr = DataLoader(gs_tr, batch_size=cfg['bs'], shuffle=True)
    L_va = DataLoader(gs_va, batch_size=BS_EV,     shuffle=False)
    best, wait = -1.0, 0
    for ep in range(1, EPOCHS + 1):
        model.train()
        for b in L_tr:
            b = b.to(DEV); opt.zero_grad()
            crit(model(b.x, b.edge_index, b.batch), b.y.view(-1).float()).backward()
            opt.step()
        v = evaluate(model, L_va, full=False)['auroc']
        if v > best: best, wait = v, 0
        else:        wait += 1
        if wait >= PATIENCE: break
    return best

def eval_cfg(cfg, ckpt, n_fold=5, seed=SEED, ep_name='Broad'):
    """검증 AUROC의 fold 평균. 테스트 fold 는 열지 않는다."""
    gs, lab, scaf = subset(ep_name)
    folds = scaffold_folds(lab, scaf, n_fold, seed)
    vals = []
    for k, (tr, te) in enumerate(folds):
        inner = strat_folds(lab[tr], 5, seed)[0]
        tr2, va = tr[inner[0]], tr[inner[1]]
        vals.append(fit_cfg([gs[i] for i in tr2], lab[tr2],
                            [gs[i] for i in va],  lab[va], ckpt, seed * 100 + k, cfg))
    return float(np.mean(vals)), float(np.std(vals))

In [5]:
# ── 셀 3 │ 한 번에 하나씩 바꾸는 탐색 (base + 8 조합) ──
CKPT_FILE = NEW + '03_dedup/pretrain_drop.pt'   # 외부·arm 비교와 같은 백본
assert os.path.exists(CKPT_FILE), CKPT_FILE

GRID = {'lr_bb':  [3e-4, 1e-3],
        'lr_clf': [1e-3, 1e-2],
        'drop':   [0.2, 0.5],
        'wd':     [1e-3],
        'bs':     [64]}

cfgs = [('base (used in the paper)', dict(BASE))]
for k, vs in GRID.items():
    for v in vs:
        c = dict(BASE); c[k] = v
        cfgs.append((f'{k} = {v:g}', c))

rows = []
for name, c in cfgs:
    t0 = datetime.datetime.now()
    m, s = eval_cfg(c, CKPT_FILE)
    mins = (datetime.datetime.now() - t0).total_seconds() / 60
    rows.append({'Configuration': name, **c, 'val_AUROC': m, 'sd': s})
    print(f'{name:<28} val AUROC {m:.4f} ± {s:.4f}   ({mins:.1f}분)')

H = pd.DataFrame(rows)
os.makedirs(OUT, exist_ok=True)
H.to_csv(OUT + 'hparam_search.csv', index=False, encoding='utf-8-sig')
print('\n저장:', OUT + 'hparam_search.csv')

base (used in the paper)     val AUROC 0.6879 ± 0.0192   (2.5분)
lr_bb = 0.0003               val AUROC 0.6959 ± 0.0185   (2.6분)
lr_bb = 0.001                val AUROC 0.7042 ± 0.0165   (2.3분)
lr_clf = 0.001               val AUROC 0.6815 ± 0.0201   (3.2분)
lr_clf = 0.01                val AUROC 0.6916 ± 0.0241   (2.4분)
drop = 0.2                   val AUROC 0.6980 ± 0.0098   (3.2분)
drop = 0.5                   val AUROC 0.6739 ± 0.0252   (1.9분)
wd = 0.001                   val AUROC 0.6866 ± 0.0187   (2.2분)
bs = 64                      val AUROC 0.6846 ± 0.0155   (1.4분)

저장: C:/Users/iisl/Desktop/0_Molecualar_Informatics/2026_final/2026_new/results/finetune/hparam_search.csv


In [6]:
# ── 셀 4 │ SI 용 표 ──
H = pd.read_csv(OUT + 'hparam_search.csv', encoding='utf-8-sig')
base = H.loc[H['Configuration'].str.startswith('base'), 'val_AUROC'].iloc[0]
H['Δ vs base'] = H['val_AUROC'] - base
show = H[['Configuration', 'val_AUROC', 'sd', 'Δ vs base']].copy()
for c in ['val_AUROC', 'sd', 'Δ vs base']:
    show[c] = show[c].map(lambda x: f'{x:+.4f}' if c == 'Δ vs base' else f'{x:.4f}')
print(show.to_string(index=False))
print(f'\n최대 |Δ| = {H["Δ vs base"].abs().max():.4f}')

           Configuration val_AUROC     sd Δ vs base
base (used in the paper)    0.6879 0.0192   +0.0000
          lr_bb = 0.0003    0.6959 0.0185   +0.0080
           lr_bb = 0.001    0.7042 0.0165   +0.0163
          lr_clf = 0.001    0.6815 0.0201   -0.0064
           lr_clf = 0.01    0.6916 0.0241   +0.0037
              drop = 0.2    0.6980 0.0098   +0.0102
              drop = 0.5    0.6739 0.0252   -0.0140
              wd = 0.001    0.6866 0.0187   -0.0013
                 bs = 64    0.6846 0.0155   -0.0033

최대 |Δ| = 0.0163


AttributeError: module 'tdc' has no attribute '__version__'